# Document-level model behaviour

Reference coverage, position, dispersion, and marker types in the held-out document predictions. The structural analyses use the pre-filtered predictions.

In [ ]:
from shared import paths
from IPython.display import display


## Setup

In [ ]:
from document_classification.structure import extract_relevant_text_and_offset

paths.prepare_directories()


## Reference coverage

In [ ]:
from pathlib import Path
import pandas as pd
import pickle
import numpy as np
import re
import matplotlib.pyplot as plt


def merge_intervals(intervals):
    if not intervals:
        return []
    sorted_intervals = sorted(intervals, key=lambda x: x[0])
    merged = [sorted_intervals[0]]
    for current_start, current_end in sorted_intervals[1:]:
        last_start, last_end = merged[-1]
        if current_start <= last_end:
            merged[-1] = (last_start, max(last_end, current_end))
        else:
            merged.append((current_start, current_end))
    return merged
csv_path = Path(str(paths.document_run() / 'general/all_folds_predictions.csv'))
letters_path = Path(str(paths.CORPUS_PICKLE))
results_df = pd.read_csv(csv_path)
with open(letters_path, 'rb') as f:
    annotated_letters = pickle.load(f)
results_df['letter_id'] = results_df['letter_id'].str.strip().str.lower()
results_map = dict(zip(results_df['letter_id'], zip(results_df['true_label'], results_df['combined_prediction'])))
true_ref_ids = {
    l.id.strip().lower()
     for l in annotated_letters if hasattr(l, 'intertextual_traces') and l.intertextual_traces
}
pred_ref_ids = set(results_df['letter_id'].unique())
missing_in_preds = true_ref_ids - pred_ref_ids
print(f' Total reference letters in annotated corpus: {len(true_ref_ids)}')
print(f' Letters with predictions: {len(pred_ref_ids)}')
print(
    f' Missing from predictions: {len(missing_in_preds)} ({len(missing_in_preds) / len(true_ref_ids):.1%})',
)
if missing_in_preds:
    print(' Example missing IDs:', list(sorted(missing_in_preds))[:10])
records = []
for letter in annotated_letters:
    letter_id = letter.id.strip().lower()
    if letter_id not in results_map:
        continue
    true_label, pred_label = results_map[letter_id]
    if true_label != 1:
        continue
    try:
        relevant_text, offset_shift = extract_relevant_text_and_offset(letter.anno_text)
    except ValueError:
        continue
    text_length = len(relevant_text)
    if text_length == 0:
        continue
    intervals = []
    for trace in letter.intertextual_traces:
        for ann in trace.annotations:
            start = ann.start_offset - offset_shift
            end = ann.end_offset - offset_shift
            if end <= start or end > text_length:
                continue
            intervals.append((start, end))
    merged_intervals = merge_intervals(intervals)
    ref_chars = sum((end - start for start, end in merged_intervals))
    ratio = ref_chars / text_length if text_length > 0 else 0
    classification = 'FN' if true_label == 1 and pred_label == 0 else 'TP'
    records.append(
        {'letter_id': letter_id, 'reference_ratio': ratio, 'classification': classification},
    )
df = pd.DataFrame(records)
print(f' Collected {len(df)} letters with merged reference coverage information.')
bins = np.linspace(0, 1, 11)
bin_labels = [f'{int(bins[i] * 100)}–{int(bins[i + 1] * 100)}%' for i in range(len(bins) - 1)]
df['ratio_bin'] = pd.cut(df['reference_ratio'], bins=bins, labels=bin_labels, include_lowest=True)
bin_counts = df.groupby('ratio_bin', observed=True)['classification'].value_counts().unstack(fill_value=0)
bin_counts['total'] = bin_counts.sum(axis=1)
bin_counts['FNR'] = bin_counts['FN'] / bin_counts['total']
bin_counts = bin_counts.reindex(bin_labels)
mean_fnr = bin_counts['FNR'].mean()
print(f' Mean False Negative Rate (across all bins): {mean_fnr:.3f}')
plt.figure(figsize=(10, 5))
bars = plt.bar(bin_counts.index, bin_counts['FNR'], color='lightcoral', edgecolor='black')
plt.axhline(y=bin_counts['FNR'].mean(), color='gray', linestyle='--', alpha=0.7, label='Mean FNR')
plt.ylim(0, 0.25)
plt.xlabel('Reference Coverage in Letter Text')
plt.ylabel('False Negative Rate')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


## Reference position

In [ ]:
from pathlib import Path
import pandas as pd
import pickle
import numpy as np
import re
import matplotlib.pyplot as plt

csv_path = Path(str(paths.document_run() / 'general/all_folds_predictions.csv'))
letters_path = Path(str(paths.CORPUS_PICKLE))
results_df = pd.read_csv(csv_path)
with open(letters_path, 'rb') as f:
    annotated_letters = pickle.load(f)
results_map = dict(zip(results_df['letter_id'], zip(results_df['true_label'], results_df['combined_prediction'])))
bins = np.linspace(0, 1, 11)
zone_labels = [f'{int(bins[i + 1] * 100)}%' for i in range(len(bins) - 1)]
records = []
for letter in annotated_letters:
    letter_id = letter.id
    if letter_id not in results_map:
        continue
    true_label, pred_label = results_map[letter_id]
    is_fn = int(true_label == 1 and pred_label == 0)
    is_tp = int(true_label == 1 and pred_label == 1)
    if not (is_fn or is_tp):
        continue
    try:
        relevant_text, offset_shift = extract_relevant_text_and_offset(letter.anno_text)
    except ValueError:
        continue
    text_length = len(relevant_text)
    if text_length == 0:
        continue
    seen_positions = set()
    for trace in letter.intertextual_traces:
        for ann in trace.annotations:
            start = ann.start_offset - offset_shift
            end = ann.end_offset - offset_shift
            if end <= start or end > text_length:
                continue
            if (start, end) in seen_positions:
                continue
            seen_positions.add((start, end))
            pos = (start + end) / 2 / text_length
            records.append({'letter_id': letter_id, 'position': pos, 'is_fn': is_fn})
df_refs = pd.DataFrame(records)
print(
    f" Collected {len(df_refs)} unique reference instances from {df_refs['letter_id'].nunique()} letters.",
)
df_refs['zone'] = pd.cut(df_refs['position'], bins=bins, labels=zone_labels, include_lowest=True)
df_grouped = df_refs.groupby('zone', observed=True).agg(total_refs=('is_fn', 'count'), fn_refs=('is_fn', 'sum')).reset_index()
df_grouped['FNR'] = df_grouped['fn_refs'] / df_grouped['total_refs']
print(df_grouped)
plt.figure(figsize=(10, 5))
plt.bar(df_grouped['zone'], df_grouped['FNR'], color='lightcoral', edgecolor='black')
plt.ylim(0, 0.25)
plt.xlabel('Reference Position in Letter')
plt.ylabel('False Negative Rate')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


## Reference dispersion

In [ ]:
from pathlib import Path
import pandas as pd
import pickle
import numpy as np
import re
import itertools
import matplotlib.pyplot as plt

csv_path = Path(str(paths.document_run() / 'general/all_folds_predictions.csv'))
letters_path = Path(str(paths.CORPUS_PICKLE))
results_df = pd.read_csv(csv_path)
with open(letters_path, 'rb') as f:
    annotated_letters = pickle.load(f)
results_map = dict(zip(results_df['letter_id'], zip(results_df['true_label'], results_df['combined_prediction'])))
records = []
for letter in annotated_letters:
    letter_id = letter.id
    if letter_id not in results_map:
        continue
    true_label, pred_label = results_map[letter_id]
    is_fn = int(true_label == 1 and pred_label == 0)
    is_tp = int(true_label == 1 and pred_label == 1)
    if not (is_fn or is_tp):
        continue
    try:
        relevant_text, offset_shift = extract_relevant_text_and_offset(letter.anno_text)
    except ValueError:
        continue
    text_length = len(relevant_text)
    if text_length == 0:
        continue
    seen_positions = set()
    positions = []
    for trace in letter.intertextual_traces:
        for ann in trace.annotations:
            start = ann.start_offset - offset_shift
            end = ann.end_offset - offset_shift
            if end <= start or end > text_length:
                continue
            if (start, end) in seen_positions:
                continue
            seen_positions.add((start, end))
            pos = start / text_length
            positions.append(pos)
    if len(positions) > 1:
        pairs = list(itertools.combinations(positions, 2))
        mean_dist = np.mean([abs(a - b) for a, b in pairs])
        records.append({'letter_id': letter_id, 'dispersion': mean_dist, 'is_fn': is_fn})
df_disp = pd.DataFrame(records)
print(f' Calculated dispersion for {len(df_disp)} letters with >1 reference.')
print(df_disp.head())


## Dispersion summary

In [ ]:
summary = df_disp.groupby('is_fn')['dispersion'].agg(['mean', 'std', 'count']).rename(
    index={0: 'TP', 1: 'FN'},
)
print('\n Dispersion summary by classification outcome:')
for label, row in summary.iterrows():
    print(f"{label}: M = {row['mean']:.3f}, SD = {row['std']:.3f}, N = {int(row['count'])}")


## Dispersion plot

In [ ]:
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.violinplot(
    data=df_disp,
    x='is_fn',
    y='dispersion',
    inner='box',
    palette=['lightcoral', 'lightgrey'],
)
plt.xticks([0, 1], ['True Positive (TP)', 'False Negative (FN)'])
plt.ylabel('Mean Pairwise Distance (Dispersion)')
plt.xlabel('')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


## Reference markers

In [ ]:
import pandas as pd
import pickle

preds_path = str(paths.document_run() / 'general/all_folds_predictions.csv')
predictions_df = pd.read_csv(preds_path)
letters_path = str(paths.CORPUS_PICKLE)
with open(letters_path, 'rb') as f:
    letters = pickle.load(f)
letter_lookup = {letter.id: letter for letter in letters}
marker_data = []
for _, row in predictions_df.iterrows():
    letter_id = row['letter_id']
    true_label = row['true_label']
    prediction = row['combined_prediction']
    classification = 'TP' if true_label == 1 and prediction == 1 else 'FN' if true_label == 1 and prediction == 0 else 'FP' if true_label == 0 and prediction == 1 else 'TN'
    letter = letter_lookup.get(letter_id)
    if not letter:
        continue
    markers = set()
    for trace in letter.intertextual_traces:
        if hasattr(trace, 'markers') and isinstance(trace.markers, list):
            markers.update(trace.markers)
    for marker in markers:
        marker_data.append({'id': letter_id, 'marker': marker, 'classification': classification})
marker_df = pd.DataFrame(marker_data)
positive_letters = predictions_df[predictions_df['true_label'] == 1].copy()
ids_with_markers = set(marker_df['id'])
unmarked_ids = set(positive_letters['letter_id']) - ids_with_markers
unmarked_rows = predictions_df[predictions_df['letter_id'].isin(unmarked_ids)].copy()
unmarked_rows['classification'] = unmarked_rows.apply(
    lambda row: 'TP' if row['true_label'] == 1 and row['combined_prediction'] == 1 else 'FN',
    axis=1,
)
unmarked_entries = pd.DataFrame(
    {
        'id': unmarked_rows['letter_id'],
        'marker': 'unmarked',
        'classification': unmarked_rows['classification'],
    },
)
marker_df = pd.concat([marker_df, unmarked_entries], ignore_index=True)
display(marker_df.head())


## Marker frequencies

In [ ]:
marker_counts = marker_df.groupby(['marker', 'classification']).size().unstack(fill_value=0)
marker_counts['total'] = marker_counts.sum(axis=1)
marker_counts = marker_counts.sort_values(by='TP', ascending=False)
display(marker_counts)


## Marker outcomes

In [ ]:
grouped = marker_df.groupby(['marker', 'classification']).size().unstack(fill_value=0)
grouped['total_pos'] = grouped.get('TP', 0) + grouped.get('FN', 0)
grouped['FNR'] = grouped.get('FN', 0) / grouped['total_pos']
grouped_sorted = grouped.sort_values(by='FNR', ascending=False)
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
grouped_sorted['FNR'].plot(kind='bar', color='#e74c3c')
plt.ylabel('False Negative Rate')
plt.xlabel('Marker')
plt.ylim(0, 1.05)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y')
plt.tight_layout()
plt.savefig(str(paths.FIGURES_DIR / 'fnr_per_marker.png'), dpi=300)
plt.show()


## Title-marked references

In [ ]:
import pandas as pd
from shared.utils import load_pickle

predictions_path = str(paths.document_run() / 'general/all_folds_predictions.csv')
predictions_df = pd.read_csv(predictions_path)
letters_path = str(paths.CORPUS_PICKLE)
annotated_letters = load_pickle(letters_path)
title_marker_ids = set()
for letter in annotated_letters:
    for trace in letter.intertextual_traces:
        if 'linguistic_code_switching' in trace.markers:
            title_marker_ids.add(letter.id)
            break
title_df = predictions_df[predictions_df['letter_id'].isin(title_marker_ids)].copy()


def determine_source(row):
    if row['baseline_prediction'] == 1:
        return 'baseline'
    elif row['prediction_source'] == 'model':
        return 'model'
    else:
        return 'combined'
title_df['source'] = title_df.apply(determine_source, axis=1)


def classify_combined(row):
    if row['true_label'] == 1 and row['combined_prediction'] == 1:
        return 'TP'
    elif row['true_label'] == 1 and row['combined_prediction'] == 0:
        return 'FN'
    elif row['true_label'] == 0 and row['combined_prediction'] == 1:
        return 'FP'
    else:
        return 'TN'
title_df['classification_combined'] = title_df.apply(classify_combined, axis=1)


def classify_model_only(row):
    if row['true_label'] == 1 and row['model_predictions_full_dataset'] == 1:
        return 'TP'
    elif row['true_label'] == 1 and row['model_predictions_full_dataset'] == 0:
        return 'FN'
    elif row['true_label'] == 0 and row['model_predictions_full_dataset'] == 1:
        return 'FP'
    else:
        return 'TN'
title_df['classification_model_only'] = title_df.apply(classify_model_only, axis=1)
print('Hybrid system classification:')
print(title_df['classification_combined'].value_counts())
print('\nClassification by source (hybrid):')
print(title_df.groupby('source')['classification_combined'].value_counts())
print('\nModel-only classification:')
print(title_df['classification_model_only'].value_counts())
